<a href="https://colab.research.google.com/github/yuzonfire907/data-science-2026/blob/main/Pertemuan%203_Yustinus%20Budi%20Kristiawan_240401010299.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pertemuan 3 - Data Cleaning

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import mstats
import requests

**LOAD DATASET**


In [6]:
# URL dataset dari Google Drive (Direct Download Link)
file_id = '1LfQWProB0VjWN5q8bKuRIgn-stULfIRo'
url = f'https://drive.google.com/uc?id={file_id}'

# Memuat dataset
df = pd.read_csv(url)
print("Dataset berhasil dimuat.")
print(df.shape)

df.head()

Dataset berhasil dimuat.
(130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


**EKSPLORASI AWAL**
1. df.info() Menampilkan informasi struktur dataframe
2. df.describe() Menampilkan statistik deskriptif untuk kolom numerik
3. df.isnull() Mengecek jumlah nilai yang hilang
4. sum() Mengecek jumlah baris duplikat


In [12]:
print("\n------ Info Struktru Data -------")
df.info()


------ Info Struktru Data -------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


In [13]:
print("\n------ Statistik Deskriptif -------")
df.describe()


------ Statistik Deskriptif -------


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [17]:
print("\n------- Identifikasi Missing Values & Jumlah Baris Duplikat -------")
print(df.isnull().sum())


------- Identifikasi Missing Values & Jumlah Baris Duplikat -------
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


**MENGHAPUS DATA DUPLIKAT**


Menghindari redundansi dan distorsi statistik.

In [18]:
print("Jumlah data duplikat sebelum dihapus:")
print(df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Jumlah data duplikat setelah dihapus:")
print(df.duplicated().sum())


Jumlah data duplikat sebelum dihapus:
0
Jumlah data duplikat setelah dihapus:
0


**NORMALISASI STRING**

Menyatukan variasi penulisan ke dalam satu kategori yang sama.

In [19]:
df['kota'] = df['kota'].str.strip().str.title()

df['kondisi'] = df['kondisi'].str.strip().str.lower()

df[['kota', 'kondisi']].head()

,kota,kondisi
0,Jogja,baik
1,Medan,bagus
2,Depok,baik
3,Ygy,baik
4,Medan,sedang


**IMPUTASI MISSING VALUES**

* Median untuk data numerik
* Modus untuk data kategorik



In [22]:
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())

df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

df.isnull().sum()

,0
id,0
luas_m2,0
harga_juta,0
kota,0
kamar,0
tahun_bangun,0
kondisi,0


**MENANGANI OUTLIER DENGAN IQR FENCE**

In [23]:
# 1. Tentukan fungsi pembersih
def bersihkan_outlier(df, kolom):
    # Kuartil 1 (25%) dan Kuartil 3 (75%)
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)

    # Jarak antar kuartil (IQR)
    IQR = Q3 - Q1

    # "Pagar" batas bawah dan batas atas
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    df[kolom] = df[kolom].clip(lower=batas_bawah, upper=batas_atas)

    return df

#Gunakan fungsi pada kolom yang bermasalah
df = bersihkan_outlier(df, 'harga_juta')
df = bersihkan_outlier(df, 'luas_m2')

print("Outlier telah disesuaikan menggunakan metode IQR.")

Outlier telah disesuaikan menggunakan metode IQR.


Setelah menerapkan IQR Fence, distribusi data pada kolom harga dan luas bangunan menjadi lebih stabil. Nilai-nilai ekstrem yang sebelumnya muncul akibat kesalahan input atau anomali data telah disesuaikan ke ambang batas atas dan bawah yang diizinkan, sehingga dataset siap digunakan untuk analisis statistik lebih lanjut.

**VALIDASI AKHIR**



1. Memastikan Integritas Data (Data Integrity)
2. Memastikan Keunikan Baris
3. Verifikasi Keberhasilan Logika Program
4. Audit Trail (Rekam Jejak)



In [25]:
print("Total missing values:")
print(df.isnull().sum().sum())

print("Total duplikat:")
print(df.duplicated().sum())


Total missing values:
0
Total duplikat:
0


**EXPORT DATASET**

In [26]:
df.to_csv('housing_clean.csv', index=False)
print("File 'housing_clean.csv' berhasil disimpan.")

File 'housing_clean.csv' berhasil disimpan.


**INTEGRASI API (JSONPlaceholder)**

In [28]:
url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(url, timeout=10)

if response.status_code == 200:

    data = response.json()

    api_df = pd.json_normalize(data)

    api_df[['id', 'name', 'email', 'address.city']]

else:
    print("Error:", response.status_code)

In [29]:

print(api_df.head())

   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    010-692-6593 x09125  anastasia.net      Victor Plains     Suite 879   
2         1-463-123-4447    ramiro.info  Douglas Extension     Suite 847   
3      493-170-9623 x156       kale.biz        Hoeger Mall      Apt. 692   
4          (254)954-1289   demarco.info       Skiles Walks     Suite 351   

    address.city address.zipcode address.geo.lat address.geo.lng  \
0    Gwenborough      92998-3874        -37.3159         81.1496   
1    Wisokyburgh